In [4]:
import pandas as pd

In [5]:
speech_df = pd.read_csv('PRIO_files/speeches.tsv', delimiter='\t')

In [6]:
speech_df = speech_df[speech_df['year'] <= 2020].copy()

In [7]:
speech_rmap = {
  'Turkey': 'Türkiye',
  'Ivory Coast': "Côte d'Ivoire",
  'Cape Verde': 'Cabo Verde',
  'Vietnam': 'Viet Nam',
  'Czech Republic': 'Czechia',
  'German Democratic Republic': 'East Germany',
  'Congo': 'Republic of the Congo',
}
speech_df['affiliation_cow'] = speech_df['affiliation_cow'].apply(lambda x: speech_rmap[x] if x in speech_rmap else x)

In [8]:
speech_df['cyear_id'] = speech_df['affiliation_cow'] + '_' + speech_df['year'].apply(lambda x: str(x))
unique_df = pd.DataFrame(
  speech_df['cyear_id'].unique(),
  columns=['cyear_id'],
)

In [9]:
unique_df[unique_df['cyear_id'].str.contains('Congo')]

,cyear_id
294,Republic of the Congo_1960
303,Democratic Republic of the Congo_1960
319,Democratic Republic of the Congo_1961
331,Republic of the Congo_1961
382,Republic of the Congo_1963
...,...
6361,National Episcopal Conference of the Congo_2019
6362,Democratic Republic of the Congo_2019
6456,Republic of the Congo_2019
6478,Afia Mama (Organization : Democratic Republic ...


In [10]:
# Check how many in spells we can match
spell_df = pd.read_csv('unsc_spells.csv')
spell_df

,country,y0,y1,aid0,aid1,aid_diff
0,Algeria,1968,1969,99.880000,11.280316,-88.599684
1,Algeria,1988,1989,0.156886,0.191735,0.034849
2,Algeria,2004,2005,5.574756,3.271398,-2.303358
3,Angola,2003,2004,227.583961,161.007484,-66.576477
4,Angola,2015,2016,63.266901,72.196048,8.929147
...,...,...,...,...,...,...
247,Zambia,1969,1970,0.000000,1.076593,1.076593
248,Zambia,1979,1980,99.341085,127.298409,27.957324
249,Zambia,1987,1988,45.440911,45.167538,-0.273373
250,Zimbabwe,1983,1984,143.364653,103.390929,-39.973724


In [11]:
spell_df['cyear_id'] = spell_df['country'] + '_' + spell_df['y0'].apply(lambda x: str(x))

In [12]:
spell_df

,country,y0,y1,aid0,aid1,aid_diff,cyear_id
0,Algeria,1968,1969,99.880000,11.280316,-88.599684,Algeria_1968
1,Algeria,1988,1989,0.156886,0.191735,0.034849,Algeria_1988
2,Algeria,2004,2005,5.574756,3.271398,-2.303358,Algeria_2004
3,Angola,2003,2004,227.583961,161.007484,-66.576477,Angola_2003
4,Angola,2015,2016,63.266901,72.196048,8.929147,Angola_2015
...,...,...,...,...,...,...,...
247,Zambia,1969,1970,0.000000,1.076593,1.076593,Zambia_1969
248,Zambia,1979,1980,99.341085,127.298409,27.957324,Zambia_1979
249,Zambia,1987,1988,45.440911,45.167538,-0.273373,Zambia_1987
250,Zimbabwe,1983,1984,143.364653,103.390929,-39.973724,Zimbabwe_1983


In [13]:
merged_df = spell_df.merge(
  unique_df,
  on='cyear_id',
  how='left',
  # indicator=True
)
merged_df

,country,y0,y1,aid0,aid1,aid_diff,cyear_id
0,Algeria,1968,1969,99.880000,11.280316,-88.599684,Algeria_1968
1,Algeria,1988,1989,0.156886,0.191735,0.034849,Algeria_1988
2,Algeria,2004,2005,5.574756,3.271398,-2.303358,Algeria_2004
3,Angola,2003,2004,227.583961,161.007484,-66.576477,Angola_2003
4,Angola,2015,2016,63.266901,72.196048,8.929147,Angola_2015
...,...,...,...,...,...,...,...
247,Zambia,1969,1970,0.000000,1.076593,1.076593,Zambia_1969
248,Zambia,1979,1980,99.341085,127.298409,27.957324,Zambia_1979
249,Zambia,1987,1988,45.440911,45.167538,-0.273373,Zambia_1987
250,Zimbabwe,1983,1984,143.364653,103.390929,-39.973724,Zimbabwe_1983


In [14]:
# merged_df['_merge'].value_counts()

In [15]:
# merged_df[merged_df['_merge'] == 'left_only']

Cool. Now we can merge the other way, since we know that all countries with a spell spoke at least once!

In [16]:
speech_aid_df = speech_df.merge(
  spell_df, on='cyear_id', how='inner',
)
speech_aid_df

,Unnamed: 0,speech_id,record_id,doc_name,meeting_num,year,month,day,topic,agenda,...,igo,un_org,ngo,cyear_id,country,y0,y1,aid0,aid1,aid_diff
0,2,SC00001-01-003,SC00001-01,S_PV.1,1,1946,1,17,Opening of the meeting by the Temporary Chairm...,Opening of the meeting by the temporary Chair...,...,0,0,0,Poland_1946,Poland,1946,1947,2449.891067,1737.721022,-712.170045
1,11,SC00001-01-012,SC00001-01,S_PV.1,1,1946,1,17,Opening of the meeting by the Temporary Chairm...,Opening of the meeting by the temporary Chair...,...,0,0,0,Brazil_1946,Brazil,1946,1947,187.363835,12.770138,-174.593697
2,12,SC00001-01-013,SC00001-01,S_PV.1,1,1946,1,17,Opening of the meeting by the Temporary Chairm...,Opening of the meeting by the temporary Chair...,...,0,0,0,Poland_1946,Poland,1946,1947,2449.891067,1737.721022,-712.170045
3,87,SC00002-01-062,SC00002-01,S_PV.2,2,1946,1,25,Adoption of directive to the Military Staff Co...,Adoption of the directive to the Military Staf...,...,0,0,0,Poland_1946,Poland,1946,1947,2449.891067,1737.721022,-712.170045
4,99,SC00003-01-011,SC00003-01,S_PV.3,3,1946,1,28,Filming of proceedings;The Iranian Question,Letter from the Head of the Iranian delegation...,...,0,0,0,Poland_1946,Poland,1946,1947,2449.891067,1737.721022,-712.170045
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25902,140928,SC08696-01-011,SC08696-01,S_PV.8696,8696,2019,12,20,The situation in the Middle East,The situation in the Middle East,...,0,0,0,South Africa_2019,South Africa,2019,2020,631.945424,1114.214584,482.269160
25903,140931,SC08696-01-014,SC08696-01,S_PV.8696,8696,2019,12,20,The situation in the Middle East,The situation in the Middle East,...,0,0,0,Indonesia_2019,Indonesia,2019,2020,266.279029,177.840499,-88.438530
25904,140945,SC08697-01-007,SC08697-01,S_PV.8697,8697,2019,12,20,The situation in the Middle East,The situation in the Middle East,...,0,0,0,Dominican Republic_2019,Dominican Republic,2019,2020,57.902418,74.615644,16.713226
25905,140956,SC08697-01-018,SC08697-01,S_PV.8697,8697,2019,12,20,The situation in the Middle East,The situation in the Middle East,...,0,0,0,Indonesia_2019,Indonesia,2019,2020,266.279029,177.840499,-88.438530


In [18]:
speech_aid_df.to_parquet('speech_aid.parquet', index=False)

downloading https://repo1.maven.org/maven2/com/google/http-client/google-http-client-jackson2/1.43.0/google-http-client-jackson2-1.43.0.jar ...
	[SUCCESSFUL ] com.google.http-client#google-http-client-jackson2;1.43.0!google-http-client-jackson2.jar (132ms)
downloading https://repo1.maven.org/maven2/com/google/http-client/google-http-client-gson/1.43.0/google-http-client-gson-1.43.0.jar ...
	[SUCCESSFUL ] com.google.http-client#google-http-client-gson;1.43.0!google-http-client-gson.jar (128ms)


Cool. Now the sentiment analysis part

26/11/26 12:20:43 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
